# XGBoost + SHAP on NOAA ocean-water data (Phase 5b)
Run the cells **top to bottom** (Runtime > Run all also works).
Upload `NOAA_water_pieces_per_m3.csv` when Cell 2 asks. Same target, features and spatial block CV as `05_ml_experiments.py`.

## Cell 1: install libraries

In [ ]:
!pip -q install -U xgboost shap
import xgboost, shap, sklearn
print('xgboost', xgboost.__version__, '| shap', shap.__version__, '| sklearn', sklearn.__version__)

## Cell 2: upload the CSV (click 'Choose files', select NOAA_water_pieces_per_m3.csv)

In [ ]:
from google.colab import files
up = files.upload()
print(list(up.keys()))

## Cell 3: random vs spatial-block cross-validation with XGBoost

In [ ]:
import pandas as pd, numpy as np, xgboost as xgb
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FILE = 'NOAA_water_pieces_per_m3.csv'
df = pd.read_csv(FILE, low_memory=False)
LAT, LON = 'Latitude (degree)', 'Longitude (degree)'
need = [LAT, LON, 'year', 'month', 'Sampling Method', 'Mesh Size (mm)', 'Water Sample Depth (m)', 'value']
missing = [c for c in need if c not in df.columns]
assert not missing, f'missing columns: {missing}'
d = df.dropna(subset=[LAT, LON, 'year', 'value']).copy()
d['y'] = np.log10(d['value'] + 1e-3)
d['block'] = np.floor(d[LAT] / 10).astype(int).astype(str) + '_' + np.floor(d[LON] / 10).astype(int).astype(str)
print('records used:', len(d), '| blocks:', d.block.nunique())   # expect 16987 and 344

num = [LAT, LON, 'year', 'month', 'Mesh Size (mm)', 'Water Sample Depth (m)']; cat = ['Sampling Method']
def make():
    pre = ColumnTransformer([('num', SimpleImputer(strategy='median'), num),
                             ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), cat)], sparse_threshold=0)
    return Pipeline([('pre', pre), ('m', xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                     subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1))])
X = d[num + cat]
rows = []
for name, sp, g in [('random 5-fold', KFold(5, shuffle=True, random_state=42), None),
                    ('spatial block 5-fold', GroupKFold(5), d['block'])]:
    oof = np.zeros(len(d)); fold_r2 = []
    for tr, te in sp.split(X, d['y'], g):
        oof[te] = make().fit(X.iloc[tr], d['y'].iloc[tr]).predict(X.iloc[te])
        fold_r2.append(r2_score(d['y'].iloc[te], oof[te]))
    rows.append(dict(validation=name, MAE=mean_absolute_error(d.y, oof), RMSE=mean_squared_error(d.y, oof) ** .5,
                     R2=r2_score(d.y, oof), R2_fold_min=min(fold_r2), R2_fold_max=max(fold_r2)))
res = pd.DataFrame(rows).round(3)
print(res.to_string(index=False))
res.to_csv('xgboost_results.csv', index=False)

## Cell 4: SHAP explanation on held-out spatial folds (explains the model, NOT causal)

In [ ]:
import shap, matplotlib.pyplot as plt
shap_vals, Xs = [], []
for tr, te in GroupKFold(5).split(X, d['y'], d['block']):
    m = make().fit(X.iloc[tr], d['y'].iloc[tr])
    Xt = m.named_steps['pre'].transform(X.iloc[te]); names = m.named_steps['pre'].get_feature_names_out()
    try:
        sv = shap.TreeExplainer(m.named_steps['m']).shap_values(Xt)
    except Exception as e:                      # fallback if shap/xgboost versions disagree
        print('TreeExplainer failed, using xgboost native contributions:', type(e).__name__)
        sv = m.named_steps['m'].get_booster().predict(xgb.DMatrix(Xt, feature_names=list(names)), pred_contribs=True)[:, :-1]
    shap_vals.append(sv); Xs.append(Xt)
SV = np.vstack(shap_vals); XT = np.vstack(Xs)
imp = pd.Series(np.abs(SV).mean(0), index=names).sort_values(ascending=False)
print(imp.head(12).round(4).to_string())
imp.to_csv('shap_mean_abs_importance.csv')
plt.figure(); shap.summary_plot(SV, XT, feature_names=list(names), show=False, max_display=12)
plt.tight_layout(); plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight'); plt.show()

## Cell 5: download the results (3 files)

In [ ]:
from google.colab import files
for f in ['xgboost_results.csv', 'shap_mean_abs_importance.csv', 'shap_summary.png']:
    files.download(f)

**Then:** copy the table printed in Cell 3 and the top-12 list printed in Cell 4 and send them back.